In [2]:

import h5py
import illustris_python as il
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter, maximum_filter
from pathlib import Path

In [3]:
finalcands = [188906, 119466, 101509]
basePath = '../sims.TNG/TNG50-1/output'

snapnum = 91
subhalofields = ['SubhaloMass', 'SubhaloMassType',
                 'SubhaloPos', 'SubhaloGrNr', 'SubhaloHalfmassRadType', 'SubhaloFlag']
subhalos = il.groupcat.loadSubhalos(basePath, snapnum, fields=subhalofields)
subhalos.keys()

dict_keys(['count', 'SubhaloMass', 'SubhaloMassType', 'SubhaloPos', 'SubhaloGrNr', 'SubhaloHalfmassRadType', 'SubhaloFlag'])

In [4]:
def get_sfr(finalcands, radius=100, chunk_size = 50000):
    print('hi')
    h = 0.6774
    a = 1 / 1.1
    boxsize = 35000.0
    galaxy_ids = [int(gid) for gid in finalcands]
    galaxy_ids = list(dict.fromkeys(galaxy_ids))

    # Galaxy centre in native TNG coordinates (ckpc/h)
    centres = {
        gid: np.asarray(
            subhalos["SubhaloPos"][gid],
            dtype=np.float64
        )
        for gid in galaxy_ids
    }

    # Store selected gas for each galaxy
    results = {gid: [] for gid in galaxy_ids}

    # Find snapshot files
    snapshot_dir = Path(basePath) / f"snapdir_{snapnum:03d}"

    snapshot_files = sorted(
        snapshot_dir.glob(f"snap_{snapnum:03d}.*.hdf5"),
        key=lambda p: int(p.name.split(".")[-2])
    )

    if not snapshot_files:
        raise FileNotFoundError(
            f"No snapshot files found in {snapshot_dir}"
        )

    print(f"Searching around {len(galaxy_ids)} galaxies")
    print(f"Snapshot files: {len(snapshot_files)}")

    for file_number, filename in enumerate(snapshot_files):

        with h5py.File(filename, "r") as f:

            if "PartType0" not in f:
                continue

            gas = f["PartType0"]
            count = gas["Coordinates"].shape[0]

            for start in range(0, count, chunk_size):

                end = min(start + chunk_size, count)

                # Read only gas positions and SFR initially
                sfr = gas["StarFormationRate"][start:end]
                forming = sfr > 0

                if not np.any(forming):
                    continue

                sf_pos = gas["Coordinates"][start:end][forming]
                sf_sfr = sfr[forming]
                sf_masses = None

                # Check the same gas cells against all galaxies
                for gid in galaxy_ids:

                    relative_pos = sf_pos.astype(
                        np.float64, copy=True
                    )

                    relative_pos -= centres[gid]

                    # Periodic boundary correction
                    relative_pos -= boxsize * np.round(
                        relative_pos / boxsize
                    )

                    # Convert to physical kpc
                    relative_pos *= a / h

                    # Select gas within the spherical region
                    r2 = np.sum(relative_pos**2, axis=1)
                    nearby = r2 < radius**2

                    if not np.any(nearby):
                        continue

                    # Only now read masses for the matching chunk
                    # (cached once per chunk if multiple galaxies match)
                    if sf_masses is None:
                        sf_masses = gas["Masses"][start:end][forming]

                    selected = pd.DataFrame({
                        "x": relative_pos[nearby, 0],
                        "y": relative_pos[nearby, 1],
                        "z": relative_pos[nearby, 2],
                        "Mass": sf_masses[nearby] * 1e10 / h,
                        "SFR": sf_sfr[nearby]
                    })

                    results[gid].append(selected)


        print(
            f"Processed {file_number + 1}/{len(snapshot_files)} files",
            flush=True)

    # Combine the results after scanning the entire snapshot
    columns = ["x", "y", "z", "Mass", "SFR"]

    sf_gasses = {}

    for gid in galaxy_ids:

        if results[gid]:
            sf_gasses[gid] = pd.concat(
                results[gid],
                ignore_index=True
            )
        else:
            sf_gasses[gid] = pd.DataFrame(columns=columns)

    # Print summary
    for gid, df in sf_gasses.items():

        print(f"\nSubhalo {gid}")
        print("Star-forming gas cells:", len(df))
        print(f"Total SFR: {df['SFR'].sum():.6f} Msun/yr")

    return sf_gasses

sf_gasses = get_sfr(finalcands)

hi
Searching around 3 galaxies
Snapshot files: 680
Processed 1/680 files
Processed 2/680 files
Processed 3/680 files
Processed 4/680 files
Processed 5/680 files
Processed 6/680 files
Processed 7/680 files
Processed 8/680 files
Processed 9/680 files
Processed 10/680 files
Processed 11/680 files
Processed 12/680 files
Processed 13/680 files
Processed 14/680 files
Processed 15/680 files
Processed 16/680 files
Processed 17/680 files
Processed 18/680 files
Processed 19/680 files
Processed 20/680 files
Processed 21/680 files
Processed 22/680 files
Processed 23/680 files
Processed 24/680 files
Processed 25/680 files
Processed 26/680 files
Processed 27/680 files
Processed 28/680 files
Processed 29/680 files
Processed 30/680 files
Processed 31/680 files
Processed 32/680 files
Processed 33/680 files
Processed 34/680 files
Processed 35/680 files
Processed 36/680 files
Processed 37/680 files
Processed 38/680 files
Processed 39/680 files
Processed 40/680 files
Processed 41/680 files
Processed 42/68

In [5]:
from pathlib import Path

output_dir = Path("sfr_results")
output_dir.mkdir(exist_ok=True)

for gid, df in sf_gasses.items():
    df.to_csv(output_dir / f"sf_gas_{gid}.csv", index=False)

print("All SFR results saved!")

All SFR results saved!
